# Federated Time-Series Queries with chrontext

This notebook demonstrates how **maplib's built-in chrontext engine** lets you write a single SPARQL query that transparently joins a knowledge graph with a time-series database.

## What is virtualisation?

In the knowledge graph world, *virtualisation* means querying data **where it already lives** — without copying it into the graph first. Instead of extracting millions of time-series rows into RDF triples (slow, memory-hungry, and instantly stale), chrontext leaves the data in DuckDB/PostgreSQL/BigQuery and translates the relevant parts of your SPARQL query into native database calls on the fly.

Your knowledge graph captures *what things are and how they relate* (sensors, stations, grid topology), while your time-series database stores *what happened and when* (measurements, events, readings). Virtualisation lets you ask questions that span both worlds in a single query — no ETL pipeline, no stale snapshots, no glue code.

## Our scenario

12 Norwegian weather stations — from Blindern in Oslo to Hammerfest in the Arctic. Each station has sensors for temperature, wind speed, and precipitation.

| Data | Where it lives | What it contains |
|------|---------------|------------------|
| **Knowledge graph** (maplib) | In-memory RDF | Station metadata, sensor relationships, types, coordinates |
| **Time-series DB** (DuckDB) | `weather.duckdb` | Real weather forecasts from MET Norway (~9 days hourly) |

The time-series data comes from MET Norway's free [Locationforecast API](https://api.met.no/weatherapi/locationforecast/2.0/documentation) — no API key required.

> **Prerequisites:** Run `python setup_data.py` first to fetch data and generate the DuckDB database.

---
## Step 1: Build the Knowledge Graph

We map tabular station metadata into RDF triples using **OTTR templates**. The template `stations.stottr` defines two patterns:
- **`Station`**: maps each CSV row to a `wx:WeatherStation` with properties like name, municipality, coordinates, and type
- **`Sensor`**: links each sensor to its station via `wx:hasSensor` and records what it measures

In [ ]:
from maplib import Model
import polars as pl
import duckdb

ns     = "http://weather.treehouse.example/"
ns_tpl = "http://weather.treehouse.example/tpl/"
ct_ns  = "https://github.com/DataTreehouse/chrontext#"

m = Model()
m.add_template(open("tpl/stations.stottr").read())

In [ ]:
df_stations = pl.read_csv("data/stations.csv")
df_stations = df_stations.with_columns(
    (pl.lit(ns) + pl.col("station_id")).alias("station_uri")
)

m.map(ns_tpl + "Station", df_stations.select([
    "station_uri", "name", "municipality", "latitude", "longitude",
    "station_type", "elevation_m", "installed_year",
]))

measurands = ["temperature", "wind_speed", "precipitation"]
sensor_rows = []
for row in df_stations.iter_rows(named=True):
    for measurand in measurands:
        sensor_rows.append({
            "station_uri": row["station_uri"],
            "sensor_uri":  f"{ns}{row['station_id']}_sensor_{measurand}",
            "measurand":   measurand,
        })

m.map(ns_tpl + "Sensor", pl.DataFrame(sensor_rows))

# Link each sensor to its time-series in DuckDB via chrontext.
# chrontext needs three predicates per sensor:
#   sensor  --ct:hasTimeseries-->  ts_node
#   ts_node --ct:hasExternalId-->  "ST001_sensor_temperature"  (matches SQL id)
#   ts_node --ct:hasResource-->    "temperature"               (matches resource_sql_map key)
ts_link_rows = []
ts_extid_rows = []
ts_resource_rows = []

for row in df_stations.iter_rows(named=True):
    for measurand in measurands:
        sensor_id = f"{row['station_id']}_sensor_{measurand}"
        sensor_uri = f"{ns}{sensor_id}"
        ts_node = f"{ns}ts/{sensor_id}"

        ts_link_rows.append({"subject": sensor_uri, "object": ts_node})
        ts_extid_rows.append({"subject": ts_node, "object": sensor_id})
        ts_resource_rows.append({"subject": ts_node, "object": measurand})

m.map_triples(pl.DataFrame(ts_link_rows),     predicate=f"{ct_ns}hasTimeseries")
m.map_triples(pl.DataFrame(ts_extid_rows),    predicate=f"{ct_ns}hasExternalId")
m.map_triples(pl.DataFrame(ts_resource_rows), predicate=f"{ct_ns}hasResource")

print(f"Knowledge graph: {m.size()} triples ({len(df_stations)} stations, {len(sensor_rows)} sensors)")

---
## Step 2: Query the Knowledge Graph

Before we involve any time-series data, let's explore the graph with pure SPARQL.

In [ ]:
m.query(open("queries/sensor_overview.rq").read())

In [ ]:
m.query("""
    PREFIX wx: <http://weather.treehouse.example/>
    SELECT ?name ?municipality ?latitude ?longitude
    WHERE {
        ?station a wx:WeatherStation ;
                 wx:stationType \"coastal\" ;
                 rdfs:label ?name ;
                 wx:municipality ?municipality ;
                 wx:latitude ?latitude ;
                 wx:longitude ?longitude .
    }
    ORDER BY DESC(?latitude)
""")

---
## Step 3: Federated Query with chrontext

Now for the interesting part. We want to answer: **"What are the forecast temperatures at coastal stations?"**

This question spans both worlds — the knowledge graph knows which stations are coastal, and DuckDB has the temperature forecasts. chrontext lets us answer it with **one SPARQL query**.

The setup:
1. Wrap DuckDB in a **VirtualizedDatabase** with SQL mappings that link sensor IDs to measurement columns
2. Define **resource templates** that describe how SQL rows map to RDF triple patterns
3. Call `m.add_virtualization()` — then just use `m.query()` as usual

### 3a: Set up DuckDB virtualisation

The `resource_sql_map` is the key bridge: for each measurand, a SQL select produces `id` (matching the `ct:hasExternalId` value in the graph), `timestamp`, and `value` columns.

In [ ]:
from maplib import (
    VirtualizedDatabase,
    Prefix, Variable, Template, Parameter, RDFType, Triple, xsd,
)
from sqlalchemy import MetaData, Table, Column, select, literal_column

# DuckDB wrapper — chrontext needs a class with a query() → DataFrame method
class WeatherDuckDB:
    def __init__(self, path):
        self.con = duckdb.connect(path, read_only=True)
        self.con.execute("SET TimeZone = 'UTC'")

    def query(self, sql: str) -> pl.DataFrame:
        return self.con.execute(sql).pl()

db = WeatherDuckDB("data/weather.duckdb")

# SQLAlchemy metadata for the measurements table
metadata = MetaData()
measurements = Table("measurements", metadata,
    Column("station_id"),
    Column("timestamp"),
    Column("temperature"),
    Column("wind_speed"),
    Column("precipitation"),
)

# Each resource maps a measurand to its timeseries in DuckDB.
# The "id" column must match the ct:hasExternalId values in the graph.
def make_resource_sql(measurand: str):
    return select(
        measurements.c.timestamp,
        measurements.c[measurand].label("value"),
    ).select_from(measurements).add_columns(
        literal_column(f"(measurements.station_id || '_sensor_{measurand}')").label("id"),
    )

vdb = VirtualizedDatabase(
    database=db,
    resource_sql_map={name: make_resource_sql(name) for name in measurands},
    sql_dialect="postgres",
)
print("DuckDB virtualisation configured")

### 3b: Define resource templates and add virtualisation

Each resource template describes how SQL rows (id, timestamp, value) map to RDF triple patterns. Once we call `m.add_virtualization()`, `m.query()` transparently spans both the in-memory graph and DuckDB.

In [ ]:
ct = Prefix("https://github.com/DataTreehouse/chrontext#")

def make_ts_template(name: str) -> Template:
    """Create a timeseries template for a given measurand."""
    id_var        = Variable("id")
    timestamp_var = Variable("timestamp")
    value_var     = Variable("value")
    dp_var        = Variable("dp")

    return Template(
        iri=ct.suf(f"{name}TimeSeries"),
        parameters=[
            Parameter(variable=id_var,        rdf_type=RDFType.Literal(xsd.string)),
            Parameter(variable=timestamp_var, rdf_type=RDFType.Literal(xsd.dateTime)),
            Parameter(variable=value_var,     rdf_type=RDFType.Literal(xsd.double)),
        ],
        instances=[
            Triple(id_var, ct.suf("hasDataPoint"), dp_var),
            Triple(dp_var, ct.suf("hasValue"),     value_var),
            Triple(dp_var, ct.suf("hasTimestamp"), timestamp_var),
        ],
    )

m.add_virtualization(
    virtualized_database=vdb,
    resources={
        "temperature":   make_ts_template("Temperature"),
        "wind_speed":    make_ts_template("WindSpeed"),
        "precipitation": make_ts_template("Precipitation"),
    },
)
print("Chrontext virtualisation added — ready for federated queries!")

### 3c: Run the federated query

One SPARQL query that finds coastal stations in the knowledge graph **and** computes temperature statistics from DuckDB. Same `m.query()` we used for pure KG queries in Step 2:

In [ ]:
federated_result = m.query("""
    PREFIX wx:         <http://weather.treehouse.example/>
    PREFIX rdfs:       <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX ct:         <https://github.com/DataTreehouse/chrontext#>
    PREFIX xsd:        <http://www.w3.org/2001/XMLSchema#>

    SELECT ?name ?municipality (AVG(?temp) AS ?avg_temp)
                               (MIN(?temp) AS ?min_temp)
                               (MAX(?temp) AS ?max_temp)
    WHERE {
        ?station a wx:WeatherStation ;
                 rdfs:label      ?name ;
                 wx:municipality ?municipality ;
                 wx:stationType  "coastal" ;
                 wx:hasSensor    ?sensor .
        ?sensor  wx:measurand    "temperature" .

        ?sensor ct:hasTimeseries ?ts .
        ?ts ct:hasDataPoint ?dp .
        ?dp ct:hasTimestamp ?t ;
            ct:hasValue     ?temp .
    }
    GROUP BY ?name ?municipality
    ORDER BY ?avg_temp
""")

print("Coastal stations — federated query (one SPARQL, no glue code):")
federated_result

Under the hood, maplib identified which triple patterns belong to the knowledge graph and which need the time-series database, pushed the aggregation down to DuckDB, and joined the results using zero-copy Arrow DataFrames.

---
## Step 4: Explore the Time-Series Data

Let's look at what's inside DuckDB — real forecast data from MET Norway's Locationforecast API.

In [ ]:
con = duckdb.connect("data/weather.duckdb", read_only=True)

total = con.execute("SELECT COUNT(*) FROM measurements").fetchone()[0]
time_range = con.execute("""
    SELECT MIN(timestamp), MAX(timestamp) FROM measurements
""").fetchone()

print(f"Total measurements: {total:,} rows")
print(f"Time range: {time_range[0]} → {time_range[1]}")
print(f"Source: MET Norway Locationforecast 2.0 (api.met.no)")

In [ ]:
con.execute("""
    SELECT station_id,
           ROUND(MIN(temperature), 1) AS min_temp,
           ROUND(AVG(temperature), 1) AS avg_temp,
           ROUND(MAX(temperature), 1) AS max_temp,
           COUNT(*) AS data_points
    FROM measurements
    GROUP BY station_id
    ORDER BY avg_temp DESC
""").pl()

In [ ]:
con.execute("""
    SELECT timestamp, temperature, wind_speed, precipitation
    FROM measurements
    WHERE station_id = 'ST001'
    ORDER BY timestamp
    LIMIT 48
""").pl()

In [ ]:
con.close()

---
## Step 5: Export the Knowledge Graph

In [ ]:
m.write("data/stations_graph.ttl", format="turtle")
print("Written to data/stations_graph.ttl")

---
## Why this matters

This demo uses 12 stations and ~9 days of hourly forecasts — small enough to run instantly on a laptop. But the architecture scales to industrial scenarios: an electrical grid operator monitoring 50,000 sensors, a fish farming company combining water quality data with facility layouts, or an oil & gas operator linking process sensors with equipment hierarchies.

In benchmarks published in *Expert Systems and Applications*, chrontext was **10–85× faster than Ontop**. The speedup comes from predicate pushdown (aggregations execute in the DB, not Python), the Rust core (no JVM overhead), and Apache Arrow (zero-copy data exchange).

chrontext currently virtualises: **PostgreSQL**, **DuckDB**, **BigQuery**, and **OPC UA Historical Access**.

Weather data: [MET Norway Locationforecast API](https://api.met.no/weatherapi/locationforecast/2.0/documentation), freely available under the [Norwegian Licence for Open Government Data](https://data.norge.no/nlod/en/2.0).

### Links

- [maplib documentation](https://datatreehouse.github.io/maplib/)
- [maplib on GitHub](https://github.com/DataTreehouse/maplib)
- [Data Treehouse](https://www.data-treehouse.com)